In [0]:

from pyspark.sql import functions as F

SILVER_SCHEMA = "online_retail.indian_ecommerce_silver"
GOLD_SCHEMA = "online_retail.indian_ecommerce_gold"

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}
""")

# Load Silver tables
orders = spark.table(f"{SILVER_SCHEMA}.orders")
order_items = spark.table(f"{SILVER_SCHEMA}.order_items")
customers = spark.table(f"{SILVER_SCHEMA}.customers")
products = spark.table(f"{SILVER_SCHEMA}.products")
payments = spark.table(f"{SILVER_SCHEMA}.payments")
shipments = spark.table(f"{SILVER_SCHEMA}.shipments")
marketing = spark.table(f"{SILVER_SCHEMA}.marketing_campaigns")

print("Gold transformation environment initialized.")


Gold transformation environment initialized.


In [0]:

sales_performance = (
    orders
    .groupBy(
        "order_date",
        "delivery_state",
        "marketing_channel"
    )
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("final_amount").alias("total_revenue"),
        F.avg("final_amount").alias("average_order_value"),
        F.sum("shipping_fee").alias("total_shipping_fee"),
        F.sum("tax_amount").alias("total_tax")
    )
)

(
    sales_performance.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD_SCHEMA}.sales_performance")
)

display(sales_performance.limit(20))


order_date,delivery_state,marketing_channel,total_orders,total_revenue,average_order_value,total_shipping_fee,total_tax
2025-08-16,Maharashtra,SMS Marketing,1,15161.9,15161.9,0.0,722.0
2023-02-11,Kerala,Social Media - Facebook,1,4998.0,4998.0,0.0,238.0
2024-12-14,Maharashtra,Social Media - Instagram,3,23345.18,7781.7266666666665,0.0,1111.68
2025-11-23,Kerala,Direct/App,2,27477.75,13738.875,99.0,1303.75
2025-07-27,Rajasthan,Organic Search,1,3622.5,3622.5,0.0,172.5
2025-01-08,Tamil Nadu,Organic Search,2,19048.16,9524.08,0.0,907.06
2025-05-26,Uttar Pradesh,Referral,2,83747.26000000001,41873.630000000005,0.0,3987.96
2025-10-04,Kerala,Organic Search,4,85235.43,21308.8575,0.0,4058.83
2023-07-15,Haryana,Social Media - Instagram,1,36866.76,36866.76,0.0,1755.56
2024-05-26,Rajasthan,Organic Search,1,2199.12,2199.12,0.0,104.72


In [0]:

# Aggregate transactional metrics by customer

customer_order_metrics = (
    orders
    .groupBy("customer_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("final_amount").alias("lifetime_revenue"),
        F.avg("final_amount").alias("average_order_value"),
        F.max("order_date").alias("last_order_date")
    )
)

# Join customer details with transaction metrics
customer_analytics = (
    customers
    .join(
        customer_order_metrics,
        on="customer_id",
        how="left"
    )
    .select(
        customers["customer_id"],
        customers["gender"],
        customers["age_group"],
        customers["state"],
        customers["city"],
        customers["customer_segment"],
        customers["preferred_device"],
        customers["loyalty_tier"],
        F.coalesce(
            customer_order_metrics["total_orders"],
            F.lit(0)
        ).alias("total_orders"),
        F.coalesce(
            customer_order_metrics["lifetime_revenue"],
            F.lit(0.0)
        ).alias("lifetime_revenue"),
        customer_order_metrics["average_order_value"],
        customer_order_metrics["last_order_date"]
    )
)

# Save Gold table
(
    customer_analytics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{GOLD_SCHEMA}.customer_analytics"
    )
)

print("Customer analytics table created successfully.")

display(customer_analytics.limit(20))


Customer analytics table created successfully.


customer_id,gender,age_group,state,city,customer_segment,preferred_device,loyalty_tier,total_orders,lifetime_revenue,average_order_value,last_order_date
CUST100000,Male,35-44,Madhya Pradesh,Indore,Regular,Mobile App,Bronze,3,36884.57,12294.856666666667,2023-05-26
CUST100001,Male,18-24,West Bengal,Kolkata,Regular,Mobile App,Gold,7,371585.73,53083.67571428571,2024-05-23
CUST100002,Female,35-44,Gujarat,Ahmedabad,Premium,Mobile Web,Silver,10,114901.74,11490.174,2025-07-14
CUST100003,Female,25-34,Maharashtra,Pune,New,Mobile App,Bronze,1,3788.82,3788.82,2025-12-19
CUST100004,Male,35-44,Andhra Pradesh,Visakhapatnam,Budget,Desktop,Silver,2,61295.899999999994,30647.949999999997,2023-11-01
CUST100005,Male,18-24,Gujarat,Surat,New,Mobile Web,Gold,4,250948.22,62737.055,2023-08-27
CUST100006,Male,18-24,Kerala,Thiruvananthapuram,New,Mobile Web,Silver,3,78978.17000000001,26326.05666666667,2023-11-18
CUST100007,Male,45-54,Andhra Pradesh,Vijayawada,New,Mobile App,Bronze,0,0.0,null,null
CUST100008,Male,35-44,Gujarat,Ahmedabad,Regular,Desktop,Bronze,5,27952.16,5590.432,2025-12-24
CUST100009,Male,25-34,Haryana,Faridabad,Regular,Mobile App,Silver,3,110734.83,36911.61,2024-06-09


In [0]:
display(spark.sql(
    "SHOW TABLES IN online_retail.indian_ecommerce_gold"
))

database,tableName,isTemporary
indian_ecommerce_gold,customer_analytics,false
indian_ecommerce_gold,sales_performance,false


In [0]:

# Verify Gold tables created so far

display(
    spark.sql(
        "SHOW TABLES IN online_retail.indian_ecommerce_gold"
    )
)


database,tableName,isTemporary
indian_ecommerce_gold,customer_analytics,false
indian_ecommerce_gold,sales_performance,false


In [0]:

# Product performance analytics

product_sales = (
    order_items
    .groupBy("product_id")
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.sum("item_revenue").alias("total_revenue"),
        F.sum("item_cost").alias("total_cost"),
        F.sum("profit").alias("total_profit"),
        F.countDistinct("order_id").alias("total_orders")
    )
)

product_performance = (
    products
    .join(product_sales, "product_id", "left")
    .select(
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "brand",
        "price",
        "units_sold",
        "total_orders",
        "total_revenue",
        "total_cost",
        "total_profit"
    )
)

(
    product_performance.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD_SCHEMA}.product_performance")
)

display(product_performance.limit(20))

print("Product performance table created.")


product_id,product_name,category,subcategory,brand,price,units_sold,total_orders,total_revenue,total_cost,total_profit
PROD10000,Sen Basic Staple,Grocery,Staples,Sen,220.0,324,143,60964.2,56337.12,4627.080000000001
PROD10001,Gupta-Chandra Essential Men's Bottomwear,Fashion,Men's Bottomwear,Gupta-Chandra,4060.0,158,76,559325.8999999999,375329.0,183996.9
PROD10002,"Murty, Advanced Women's Casual",Footwear,Women's Casual,"Murty,",2800.0,1150,510,2816170.0,2002127.0000000007,814042.9999999994
PROD10003,Madan-Kanda Signature Bedding,Home & Kitchen,Bedding,Madan-Kanda,5680.0,261,116,1275216.8,1130304.87,144911.93
PROD10004,"Sami, Premium Home Decor",Home & Kitchen,Home Decor,"Sami,",370.0,2389,1021,772806.0499999997,599591.2200000007,173214.83000000016
PROD10005,Bhatt Essential Skincare,Beauty,Skincare,Bhatt,250.0,550,233,120920.0,83935.50000000001,36984.50000000001
PROD10006,Sen Compact Cycling,Sports,Cycling,Sen,4790.0,918,394,3856860.1,3163611.6000000006,693248.5
PROD10007,Gara-Chad Ultra Women's Ethnic Wear,Fashion,Women's Ethnic Wear,Gara-Chad,450.0,2171,927,849717.0,583629.9300000017,266087.0699999993
PROD10008,Sabharwal-Kothari Signature Snack,Grocery,Snacks,Sabharwal-Kothari,90.0,76,33,5818.95,5985.0,-166.05
PROD10009,Kapoor Pro Men's Bottomwear,Fashion,Men's Bottomwear,Kapoor,570.0,98,45,48948.75,34535.2,14413.55


Product performance table created.


In [0]:

# Shipment performance analytics

shipment_analytics = (
    shipments
    .groupBy(
        "warehouse_city",
        "shipping_method",
        "delivery_status"
    )
    .agg(
        F.countDistinct("shipment_id").alias("total_shipments"),
        F.avg("delivery_days").alias("average_delivery_days"),
        F.sum("delayed_flag").alias("delayed_shipments")
    )
    .withColumn(
        "delay_percentage",
        F.when(
            F.col("total_shipments") > 0,
            F.col("delayed_shipments") /
            F.col("total_shipments") * 100
        ).otherwise(0)
    )
)

(
    shipment_analytics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD_SCHEMA}.shipment_analytics")
)

display(shipment_analytics)

print("Shipment analytics table created.")


warehouse_city,shipping_method,delivery_status,total_shipments,average_delivery_days,delayed_shipments,delay_percentage
Chennai,Economy/Surface,Delivered,1519,null,562,36.9980250164582
Mumbai,Economy/Surface,Delivered,2256,null,846,37.5
Chennai,Standard,Delivered,4730,null,2579,54.524312896405924
Delhi-NCR (Gurugram),Same-Day,Delivered,1019,null,592,58.096172718351326
Chennai,Express,Cancelled in Transit,64,null,0,0.0
Delhi-NCR (Gurugram),Standard,Delivered,9394,null,5216,55.52480306578668
Ahmedabad,Express,Delivered,2555,null,1031,40.35225048923679
Mumbai,Next-Day,Delivered,2309,null,233,10.090948462537895
Mumbai,Standard,Delivered,6700,null,3655,54.552238805970156
Hyderabad,Express,Delivered,1889,null,723,38.274219163578614


Shipment analytics table created.


In [0]:

# Marketing campaign analytics

marketing_analytics = (
    marketing
    .withColumn(
        "click_through_rate",
        F.when(
            F.col("impressions") > 0,
            F.col("clicks") / F.col("impressions") * 100
        ).otherwise(0)
    )
    .withColumn(
        "conversion_rate",
        F.when(
            F.col("clicks") > 0,
            F.col("conversions") / F.col("clicks") * 100
        ).otherwise(0)
    )
    .withColumn(
        "return_on_ad_spend",
        F.when(
            F.col("campaign_cost") > 0,
            F.col("revenue_generated") / F.col("campaign_cost")
        ).otherwise(None)
    )
    .select(
        "campaign_id",
        "campaign_name",
        "channel",
        "campaign_type",
        "target_segment",
        "impressions",
        "clicks",
        "conversions",
        "revenue_generated",
        "campaign_cost",
        "roi",
        "click_through_rate",
        "conversion_rate",
        "return_on_ad_spend"
    )
)

(
    marketing_analytics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD_SCHEMA}.marketing_analytics")
)

display(marketing_analytics)

print("Marketing analytics table created.")


campaign_id,campaign_name,channel,campaign_type,target_segment,impressions,clicks,conversions,revenue_generated,campaign_cost,roi,click_through_rate,conversion_rate,return_on_ad_spend
CAMP1001,Direct/App Festival Special Jan 2023,Direct/App,New Product Launch,Premium,606,166,27,698500.58,97919.42,6.133,27.39273927392739,16.265060240963855,7.133422358915116
CAMP1002,Social Media - Instagram Flash Sale Jan 2023,Social Media - Instagram,Loyalty Program,Premium,18769,359,10,262727.62,83998.79,2.128,1.9127284351856784,2.785515320334262,3.12775481646819
CAMP1003,Organic Search New Product Launch Jan 2023,Organic Search,Seasonal Sale,Regular,2155,452,40,1640037.86,418495.64,2.919,20.97447795823666,8.849557522123893,3.9188887607048906
CAMP1004,SMS Marketing Festival Special Jan 2023,SMS Marketing,Festival Special,All,296,30,2,11607.02,13777.19,-0.158,10.135135135135135,6.666666666666667,0.8424809413240291
CAMP1005,Email Marketing New Product Launch Feb 2023,Email Marketing,Loyalty Program,Budget,2480,393,43,991507.43,163106.97,5.079,15.846774193548388,10.941475826972011,6.078878358171941
CAMP1006,Referral Clearance Sale Feb 2023,Referral,Clearance Sale,Budget,1622,413,69,1600898.27,347269.18,3.61,25.462392108508013,16.707021791767556,4.609963573502261
CAMP1007,Affiliate Seasonal Sale Feb 2023,Affiliate,Loyalty Program,Premium,13344,840,51,1168263.89,520021.92,1.247,6.294964028776978,6.071428571428571,2.246566625499171
CAMP1008,Social Media - Facebook Flash Sale Feb 2023,Social Media - Facebook,Clearance Sale,Budget,72338,1384,33,772654.06,708602.54,0.09,1.9132406204208023,2.384393063583815,1.0903913214875014
CAMP1009,Referral New Product Launch Mar 2023,Referral,Loyalty Program,Premium,1414,351,60,1774081.65,354421.03,4.006,24.82319660537482,17.094017094017094,5.0055767006828
CAMP1010,Paid Search New Product Launch Mar 2023,Paid Search,Loyalty Program,Budget,55744,2242,106,2900065.56,831390.79,2.488,4.021957520091848,4.727921498661909,3.488209870595271


Marketing analytics table created.


In [0]:

# Verify all Gold tables

display(
    spark.sql(
        "SHOW TABLES IN online_retail.indian_ecommerce_gold"
    )
)


database,tableName,isTemporary
indian_ecommerce_gold,customer_analytics,false
indian_ecommerce_gold,marketing_analytics,false
indian_ecommerce_gold,product_performance,false
indian_ecommerce_gold,sales_performance,false
indian_ecommerce_gold,shipment_analytics,false


In [0]:

# Final Gold layer validation

GOLD_TABLES = [
    "sales_performance",
    "customer_analytics",
    "product_performance",
    "shipment_analytics",
    "marketing_analytics"
]

gold_results = []

for table_name in GOLD_TABLES:

    full_name = f"{GOLD_SCHEMA}.{table_name}"

    df = spark.table(full_name)

    row_count = df.count()
    column_count = len(df.columns)

    gold_results.append((
        table_name,
        row_count,
        column_count,
        "PASS" if row_count > 0 else "REVIEW"
    ))

gold_validation = spark.createDataFrame(
    gold_results,
    [
        "table_name",
        "row_count",
        "column_count",
        "status"
    ]
)

display(gold_validation)

print("Gold layer validation completed.")


table_name,row_count,column_count,status
sales_performance,63446,8,PASS
customer_analytics,25000,12,PASS
product_performance,550,11,PASS
shipment_analytics,105,7,PASS
marketing_analytics,151,14,PASS


Gold layer validation completed.
